# 02 — Feature Engineering

Build the feature matrix consumed by the content tower:

| Feature | Method |
|---|---|
| `genre_vector` | Multi-hot encoding from the `genres` column |
| `description_embedding` | TF-IDF (fast) or Sentence-BERT `all-MiniLM-L6-v2` |
| `content_type_flag` | Movie=0 / TV Show=1 |
| `release_decade` | One-hot of 1920s … 2020s |
| `country_encoded` | Top-20 one-hot + 'Other' |
| `language_encoded` | Top-10 one-hot + 'Other' |
| `rating_tmdb` | Numeric TMDB score (0-10), min-max normalised |
| `popularity`, `vote_average` | Min-max normalised |
| `duration_normalized` | Min-max per content type (movies: minutes, shows: seasons) |

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd

from src.data_loader import NetflixDataLoader
from src.feature_engineering import FeatureEngineer

In [ ]:
DATA_DIR = PROJECT_ROOT / 'data'
MOVIES_CSV = DATA_DIR / 'netflix_movies_detailed_up_to_2025.csv'
TV_CSV = DATA_DIR / 'netflix_tv_shows_detailed_up_to_2025.csv'

loader = NetflixDataLoader([MOVIES_CSV, TV_CSV])
df = loader.clean(loader.load())
items = loader.get_content_features(df)
print('Items:', items.shape)
items.head(3)

In [ ]:
# Use TF-IDF by default (fast, deterministic). Flip to True to use
# Sentence-BERT (requires downloading the model the first time).
USE_SBERT = False

fe = FeatureEngineer(tfidf_max_features=512, use_sbert=USE_SBERT)
genre_matrix = fe.build_genre_matrix(items)
desc_matrix = fe.build_description_embeddings(items)
meta_matrix = fe.build_metadata_features(items)
print('genre     :', genre_matrix.shape)
print('description:', desc_matrix.shape)
print('metadata  :', meta_matrix.shape)

In [ ]:
features = fe.combine_features(genre_matrix, desc_matrix, meta_matrix)
id_map = fe.build_content_id_map(items)
print('features  :', features.shape)
print('First 5 genres in vocabulary  :', fe.genre_vocab[:5])
print('Decade vocabulary             :', fe.decade_vocab)
print('Top countries (first 5)       :', fe.country_vocab[:5])
print('Top languages (first 5)       :', fe.language_vocab[:5])

In [ ]:
# Persist artefacts for the training notebook.
out_dir = PROJECT_ROOT / 'data'
out_dir.mkdir(parents=True, exist_ok=True)
np.save(out_dir / 'features.npy', features)
np.save(out_dir / 'genre_matrix.npy', genre_matrix)
items.to_parquet(out_dir / 'items.parquet', index=False)
print('Saved:')
print('  data/features.npy       ', features.shape)
print('  data/genre_matrix.npy   ', genre_matrix.shape)
print('  data/items.parquet      ', items.shape)

**Next**: `03_model_training.ipynb` consumes `features.npy` + `genre_matrix.npy` to train the content tower under a triplet-margin objective.